# 065. next character 를 예측하여 Shakespeare Sonnet Text 생성

- 순환 신경망을 활용한 문자열 생성


- character 단위로 text 생성  

    - 문자 시퀀스 (ex. "Shakespear")가 주어지면, 시퀀스의 다음 문자("e")를 예측하는 모델을 훈련
    - 모델을 반복하여 호출하면 더 긴 텍스트 시퀀스 생성이 가능
    - 훈련이 시작되었을 때, 이 모델은 영어 단어의 철자를 모르거나 심지어 텍스트의 단위가 단어라는 것도 모름 
    
    
- google NLP tutorial 참조

In [1]:
import tensorflow as tf
import numpy as np
import time
print(tf.__version__)

2.2.0-rc3


# Text File download

- 셰익스피어 데이터셋 다운로드  


- data 는 ~/.keras/dataset 에 저장. absolute path 지정해 주어야 replace  됨. 다시 download 받을 경우 ~/.keras file 삭제 필요

In [2]:
# path_to_file = tf.keras.utils.get_file("shakespeare.txt", 
#                          "https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt")

path_to_file = tf.keras.utils.get_file("young-prince.txt", 
              "https://raw.githubusercontent.com/ironmanciti/NLP_Lecture/master/data/young_prince.txt")

114688/107468 [================================] - 0s 0us/step


In [3]:
text = open(path_to_file, 'r').read()
len(text)

46748

In [4]:
print(text[:100])

여섯 살 적에 나는 "체험한 이야기"라는 제목의, 원시림에 관한 책에서 기막힌 그림 하나를 본 적이 있다. 맹수를 집어삼키고 있는 보아 구렁이 그림이었다. 위의 그림은 그것을 옮겨


# lookup table 작성

- 문자들을 숫자로 변환


- 두 개의 조회 테이블(lookup table) 작성  
    - character to index
    - index to character


- text 중에 포함된 character 들을 이용하여 charactet-to-index, index-to-character 변환 table 작성
    - 하나는 문자를 숫자에 매핑하고 다른 하나는 숫자를 문자에 매핑  
    - 문자를 0번 인덱스부터 고유 문자 길이까지 매핑

In [0]:
chars = sorted(set(text))
nb_chars = len(chars)

char2idx = dict((c, i) for i, c in enumerate(chars))
idx2char = chars

In [6]:
print("문자 갯수 : ", nb_chars)
print()
print("character to index lookup table : \n", char2idx)
print()
print("index to character lookup table : \n",idx2char)

문자 갯수 :  851

character to index lookup table : 
 {'\n': 0, ' ': 1, '!': 2, '"': 3, '(': 4, ')': 5, ',': 6, '.': 7, '0': 8, '1': 9, '2': 10, '3': 11, '4': 12, '5': 13, '6': 14, '7': 15, '8': 16, '9': 17, '<': 18, '>': 19, '?': 20, 'B': 21, '使': 22, '儀': 23, '商': 24, '大': 25, '小': 26, '布': 27, '式': 28, '律': 29, '惑': 30, '星': 31, '的': 32, '紀': 33, '紅': 34, '詩': 35, '隊': 36, '가': 37, '각': 38, '간': 39, '갈': 40, '감': 41, '갑': 42, '값': 43, '갔': 44, '강': 45, '갖': 46, '같': 47, '개': 48, '객': 49, '거': 50, '걱': 51, '건': 52, '걷': 53, '걸': 54, '검': 55, '겁': 56, '것': 57, '게': 58, '겐': 59, '겠': 60, '겨': 61, '겪': 62, '견': 63, '결': 64, '겸': 65, '겹': 66, '겼': 67, '경': 68, '곁': 69, '계': 70, '고': 71, '곡': 72, '곤': 73, '곧': 74, '골': 75, '곰': 76, '곱': 77, '곳': 78, '공': 79, '과': 80, '관': 81, '광': 82, '괜': 83, '괴': 84, '굉': 85, '교': 86, '구': 87, '국': 88, '군': 89, '굴': 90, '굽': 91, '궁': 92, '궂': 93, '권': 94, '귀': 95, '귄': 96, '규': 97, '그': 98, '극': 99, '근': 100, '글': 101, '금': 102, '급': 103, '긋': 104, '긍': 105

### 훈련 sample 과 target 만들기 

- input data 를 고정길이로 통일하기 위해 MAX_SEQ_LEN 을 입력의 최대 길이로 지정 (임의로 결정)   


- MAX_SEQ_LEN 개의 연속된 character sequence 를 input 으로 하고, 1 character shift 하여 이어지는 동일한 길이의 character sequence 를 label 로 만든다.  

- 따라서, MAX_SEQ_LEN + 1 길이의 덩어리(chunk)로 text 를 분할하여 input, label 분리 (tf.Dataset method 사용)
        ex) MAX_SEQ_LEN = 4 인 경우,
            "Hell"  --> "ello"
            

- 이를 손쉽게 하기 위해 tensorflow 가 제공하는 tf.data.Dataset.from_tensor_slices 함수를 사용하여 text vector 를 character index 의 stream 으로 변환  

In [7]:
MAX_SEQ_LEN = 100         # time step = 100, 단일 입력에 대해 원하는 문장의 최대 글자수
examples_per_epoch = len(text) // MAX_SEQ_LEN
print("한 epoch 당 처리할 전체 sample 수 :", examples_per_epoch)

한 epoch 당 처리할 전체 sample 수 : 467


**char2index lookup table 을 이용하여 text 전체를 integer 로 변환**

In [8]:
text_as_int = np.array([char2idx[c] for c in text])
print(text_as_int)

[555 467   1 ...   7   7   7]


In [9]:
print("변환전 : {}, 변환후 : {}".format(text[:13], text_as_int[:13]))

변환전 : 여섯 살 적에 나는 "체, 변환후 : [555 467   1 454   1 630 553   1 145 189   1   3 700]


# 훈련 sample 및 target 만들기

- tf.data.Dataset.from_tensor_slices method $\rightarrow$ 주어진 tensor 들을 first dimension 을 따라 slice

In [10]:
char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)
char_dataset

<TensorSliceDataset shapes: (), types: tf.int64>

In [11]:
seq = [idx.numpy() for idx in char_dataset.take(5)]
print(seq)
print([''.join(idx2char[i]) for i in seq])

[555, 467, 1, 454, 1]
['여', '섯', ' ', '살', ' ']


  ## dataset 생성 - input text, target text

**tf.data 의 batch method 를 사용하면 개별 character 를 원하는 size 의 character sequence 로 쉽게 바꿀 수 있다**

- char_dataset 을 seq_len +1 길이의 character sequence 로 변환 (next character 를 예측하는 문제이므로 +1) 하여  
  [ : sequence]는 input 으로 사용하고 [1 : sequence+1] 은 label 로 사용  
  

- sequnces 를 입력 text 와 target text 로 분리하는 helper 함수를 작성하여 map 함수로 각 sequences 의 element 에 적용

In [0]:
# sequence 를 input, target 으로 분리
def split_input_target(chunk):
    input_text = chunk[:-1]
    target_text = chunk[1:]
    return input_text, target_text

In [13]:
dataset = char_dataset.batch(MAX_SEQ_LEN + 1, drop_remainder=True)
dataset = dataset.map(split_input_target)

input, target = next(iter(dataset))
input_seq  = [i for i in input.numpy()]
target_seq = [i for i in target.numpy()]
print(input_seq)
print(target_seq)
print("입력 data   = ", repr(''.join(idx2char[idx] for idx in input_seq)))
print("출력 data   = ", repr(''.join(idx2char[idx] for idx in target_seq)))

[555, 467, 1, 454, 1, 630, 553, 1, 145, 189, 1, 3, 700, 806, 795, 1, 604, 536, 106, 3, 280, 189, 1, 637, 368, 603, 6, 1, 591, 497, 331, 553, 1, 81, 795, 1, 692, 553, 461, 1, 106, 335, 848, 1, 98, 331, 1, 793, 145, 324, 1, 419, 1, 630, 604, 1, 613, 199, 7, 1, 350, 480, 324, 1, 665, 542, 455, 739, 71, 1, 613, 189, 1, 417, 526, 1, 87, 301, 604, 1, 98, 331, 604, 550, 199, 7, 1, 594, 603, 1, 98, 331, 599, 1, 98, 57, 600, 1, 570, 61]
[467, 1, 454, 1, 630, 553, 1, 145, 189, 1, 3, 700, 806, 795, 1, 604, 536, 106, 3, 280, 189, 1, 637, 368, 603, 6, 1, 591, 497, 331, 553, 1, 81, 795, 1, 692, 553, 461, 1, 106, 335, 848, 1, 98, 331, 1, 793, 145, 324, 1, 419, 1, 630, 604, 1, 613, 199, 7, 1, 350, 480, 324, 1, 665, 542, 455, 739, 71, 1, 613, 189, 1, 417, 526, 1, 87, 301, 604, 1, 98, 331, 604, 550, 199, 7, 1, 594, 603, 1, 98, 331, 599, 1, 98, 57, 600, 1, 570, 61, 1]
입력 data   =  '여섯 살 적에 나는 "체험한 이야기"라는 제목의, 원시림에 관한 책에서 기막힌 그림 하나를 본 적이 있다. 맹수를 집어삼키고 있는 보아 구렁이 그림이었다. 위의 그림은 그것을 옮겨'
출력 data   =  '섯 살 적에 나

# Training / Target dataset 구성

- input_sample 의 각 인덱스는 하나의 타임 스텝(time step)으로 처리  

 
- 위의 경우, time step 0 의 입력으로 모델은 "F"의 인덱스를 받고 다음 문자로 "i"의 인덱스를 예측   


- 다음 타임 스텝에서도 같은 일을 하지만 RNN은 현재 입력 문자 외에 이전 타임 스텝의 컨텍스트(context)를 고려하여 예측  


- RNN 의 context 정보는 hidden state 를 뜻함  


- 이전 batch 의 context 정보를 다음 batch 의 initial hidden state 로 받기 위해 RNN model 정의 시 stateful=True 로 지정

## Training batch 생성

- tf.data를 사용하여 train batch 생성    


- 이 데이터를 모델에 넣기 전에 데이터를 섞은 후 배치를 만든다 (다음 character 를 예측하는 모델이고 임의의 batch size 로 잘랐으므로 shuffle 필요)  


- batch method 를 이용하여 train batch 생성

In [14]:
BATCH_SIZE = 64
BUFFER_SIZE = 10000

dataset = dataset.batch(BATCH_SIZE, drop_remainder=True)
dataset

<BatchDataset shapes: ((64, 100), (64, 100)), types: (tf.int64, tf.int64)>

## Model 생성

- Many-to-Many type

<div>
<img src="https://tensorflow.org/tutorials/text/images/text_generation_training.png", width="600">
</div>

- tf.keras.Sequential model 사용  


- 3 개의 층을 사용하여 모델 정의  


    1. Embedding : 입력층, embedding_dim 차원 벡터에 각 문자의 정수 코드를 매핑하는 훈련가능한 검색 table
        - 각 character 에 대해 model 은 embedding 을 검색하고, embedding 을 입력으로 하여 GRU(LSTM) 를 1 개의 time step 으로 실행  

    2. LSTM (GRU) 를 이용한 RNN 층

    3. 완전연결층(Dense) 
        - logits 생성 : (64, 100, 65) - (batch size, MAX_SEQ_LEN, vocab_size)


- Dense layer 에서 생성된 크기가 vocab_size 인 logit 을 tf.random.categorical 에 적용하여 다음 문자의 확률분포 예측  


- stateful=True 로 지정 - 문장이 계속 연속되므로 이전 batch 의 hidden state 를 다음번 batch 의 초기 state 로 사용  


- batch_input_shape 반드시 지정 ($t_i$ 의 initial state 가 $t_{i+bs}$ 로 연결되므로)


- 입력 batch shape : (64, 100)  
- 출력 batch shape : (64, 100, 65)

In [15]:
EMBEDDING_DIM = 256
RNN_UNITS = 1024
print(nb_chars)

851


**마지막 Dense layer 의 activation='linear' --> shape(100, 65) 의 logit 생성**

- last layer 의 activation 을 softmax 로 하는 것보다 계산상 안정적임

In [0]:
def build_model(vocab_size, embedding_dim, rnn_units, batch_size):
    model = tf.keras.Sequential([
        tf.keras.layers.Embedding(vocab_size, embedding_dim, batch_input_shape=[batch_size, None]),
        tf.keras.layers.LSTM(rnn_units, 
                             return_sequences=True, 
                             stateful=True,
                             recurrent_initializer='glorot_uniform'),
        tf.keras.layers.Dense(nb_chars)   # linear activation (logit 생성)
    ])
    return model

model = build_model(vocab_size=nb_chars, embedding_dim=EMBEDDING_DIM, rnn_units=RNN_UNITS, batch_size=BATCH_SIZE)

In [19]:
model.summary()

Model: "sequential_1"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
embedding_1 (Embedding)      (64, None, 256)           217856    
_________________________________________________________________
lstm_1 (LSTM)                (64, None, 1024)          5246976   
_________________________________________________________________
dense_1 (Dense)              (64, None, 851)           872275    
Total params: 6,337,107
Trainable params: 6,337,107
Non-trainable params: 0
_________________________________________________________________


# 모델 훈련
- 이 문제는 표준적인 분류 문제로 취급 가능  


- 이전 RNN 상태와 현재 타임 스텝(time step)의 입력으로 다음 문자의 클래스를 예측   


- stateful RNN
    - time step 0 의 이전 상태는 이전 batch 의 final step 을 이어 받으므로 stateful=True 지정

## 손실함수 지정

- tf.keras.losses.sparse_categorical_crossentropy 손실 함수 이용


    - categorical_crossentropy - one-hot-encoding  
        [1,0,0]
        [0,1,0]   
        [0,0,1]  
    - sparse_categorical_crossentropy - integer encoding
        1  
        2  
        3  


- 우리의 모델은 logit 을 출력으로 반환하므로 from_logits=True 로 setting 한다.
    --> softmax activation 보다 계산이 안정적 (Tensorflow document)

In [0]:
def loss(labels, logits):
    return tf.keras.losses.sparse_categorical_crossentropy(labels, logits, from_logits=True)

model.compile(loss=loss, optimizer='adam', metrics=['accuracy'])

## checkpoint 구성

- 훈련 중 checkpoint 저장

In [0]:
import os

checkpoint_dir = './training_checkpoints'
# 체크포인트 파일 이름
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt_{epoch}")

checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(filepath=checkpoint_prefix, save_weights_only=True)

In [22]:
s = time.time()

history = model.fit(dataset, epochs=100, callbacks=[checkpoint_callback])

print(time.time() - s)

Epoch 1/100
7/7 [==============================] - 1s 81ms/step - loss: 5.7472 - accuracy: 0.1914
Epoch 2/100
7/7 [==============================] - 1s 75ms/step - loss: 4.5296 - accuracy: 0.1910
Epoch 3/100
7/7 [==============================] - 1s 76ms/step - loss: 4.4114 - accuracy: 0.2507
Epoch 4/100
7/7 [==============================] - 1s 82ms/step - loss: 4.3658 - accuracy: 0.2507
Epoch 5/100
7/7 [==============================] - 1s 76ms/step - loss: 4.3204 - accuracy: 0.2507
Epoch 6/100
7/7 [==============================] - 1s 81ms/step - loss: 4.2688 - accuracy: 0.2507
Epoch 7/100
7/7 [==============================] - 1s 75ms/step - loss: 4.2072 - accuracy: 0.2507
Epoch 8/100
7/7 [==============================] - 1s 85ms/step - loss: 4.1352 - accuracy: 0.2602
Epoch 9/100
7/7 [==============================] - 1s 74ms/step - loss: 4.0696 - accuracy: 0.2682
Epoch 10/100
7/7 [==============================] - 1s 74ms/step - loss: 4.0056 - accuracy: 0.2805
Epoch 11/100
7/7 [=

## 훈련된 model 을 이용한 text generation

- 예측 단계를 간단히 하기 위해 batch size = 1 로 변경한 새로운 model 을 rebuild 하고, last checkpoint 의 model weight 를 load  


- model rebuild 이유:

    - 우리가 작성했던 model 은 stateful=True 이므로, 작성 당시에 모델이 지정된 batch_input_shape 의  fixed batch size 만 받아 들인다.
    - 다른 batch size 에서도 수행되도록 하기 위해서는 model 을 batch_size 1 로 rebuild 하고 마지막 checkpoint 에서 저장한 model weight 를 restore

In [23]:
# batch size 1 의 new model
model = build_model(vocab_size=nb_chars, embedding_dim=EMBEDDING_DIM, rnn_units=RNN_UNITS, batch_size=1)
# weight load
model.load_weights(tf.train.latest_checkpoint(checkpoint_dir))
# batch size 1 로 rebuild
model.build(tf.TensorShape([1, None]))

model.summary()

Model: "sequential_2"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
embedding_2 (Embedding)      (1, None, 256)            217856    
_________________________________________________________________
lstm_2 (LSTM)                (1, None, 1024)           5246976   
_________________________________________________________________
dense_2 (Dense)              (1, None, 851)            872275    
Total params: 6,337,107
Trainable params: 6,337,107
Non-trainable params: 0
_________________________________________________________________


## 마지막 layer 에서 output sampling 요령

- distribution 의 argmax 를 하면 model 이 쉽게 loop 에 빠짐.  


- tf.random.categorical(logits, num_samples) 을 이용하여 categorical 분포로 부터 sample 추출  


    - logits: [batch_size, num_classes] 의 2-D Tensor. 각 slice [i, :] 는 각 class 의 normalize 되지 않은 log-probability 를 나타냄
    - num_samples: 각 row slice 에서 추출할  독립적 sample 수
    
- MAX_SEQ_LEN=100 이므로, 100 개의 timestep 별로 65 개의 character 별 확률([100, 65) 로  1 개를 random sampling 
  --> output shape [100, 1]

In [24]:
ex_batch_predictions = np.random.random((1, 10, 65))   # (batch_size, time_steps, num_chars)
# 10 개 timestep 별로 (10, 65) shape 의 확률
tf.random.categorical(ex_batch_predictions[0], num_samples=1)

<tf.Tensor: shape=(10, 1), dtype=int64, numpy=
array([[15],
       [55],
       [14],
       [23],
       [10],
       [61],
       [34],
       [50],
       [12],
       [59]])>

## 예측 loop 

### 다음 code block 에서 text 생성:

- start string 으로 시작, RNN state 초기화 및 생성할 characters 수 설정 

- start string 과 RNN state 를 이용하여 next character 의 prediction 분포를 가져옴  

- categorical 분포를 이용하여 예측된 character 의 index 계산. 예측된 character 를 model 의 next input 으로 사용  

- model 에서 반환된 RNN state 는 model 로 다시 되돌려져서 이제는 하나의 character 가 아닌 더 많은 문맥(context) 에 더해짐

- 다음 단어를 예측한 후 수정된 RNN state 가 다시 모델로 피드백되어 더 많은 context 를 얻으며 학습되는 방식임

![텍스트를 생성하기 위해 모델의 출력이 입력으로 피드백](https://tensorflow.org/tutorials/text/images/text_generation_sampling.png)

In [25]:
#start_string = "ROMEO: "
start_string = "여섯 살 적에"

num_generate = 1000   # 생성할 문자의 수

# starting_string 의 숫자화 (벡터화)
input_eval = [char2idx[c] for c in start_string]
input_eval = tf.expand_dims(input_eval, 0)
input_eval

<tf.Tensor: shape=(1, 7), dtype=int32, numpy=array([[555, 467,   1, 454,   1, 630, 553]], dtype=int32)>

### stateful RNN 이므로 새로운 text generation 을 위해 reset_states() method 를 이용하여 initial state 초기화

In [0]:
text_generated = []

model.reset_states()   # initial state

for i in range(num_generate):
    predictions = model(input_eval)    
    # batch dimension 제거
    predictions = tf.squeeze(predictions, 0)  
    # greedy(argmax) 하지 않도록 분포에서 sampling
    predicted_id = tf.random.categorical(predictions, num_samples=1)[-1, 0].numpy() # scalar
    
    # print(predictions.shape)
    # predicted = tf.random.categorical(predictions, num_samples=1)
    # print(predicted)
    # print(predicted[-1, 0])
    # break
    
    # 예측된 character 를 이전 은닉 상태와 함께 다음 입력으로 model 에 전달
    input_eval = tf.expand_dims([predicted_id], 0)   
    text_generated.append(idx2char[predicted_id])

In [27]:
print(start_string + ''.join(text_generated))

여섯 살 적에 대힌 이야기를 껐다.
  평물이 좀 대해서는 얼굴을 놀랐어. 첫중부해서 세대로 생각안지는 거야......
  "그럴 수 있겠던 섬, 이 마군 보이 내 망말을 절약을 하고 사냥한 그 바람도 뱀이 마르운 듯이다.
  그러한 왕자를 앉았다.
  "아니지만 하품을 하나 않았다.
  첫번째 말했다.
  "그것만 같은 람이 말할 수가 돌들을 꽃 둘러더  스르더  그 조우심은 무서 그렸다.그것은 모두 걸의 그마 친구로 주는건 듯과 떠나가 거야. 대답을하고 했으므. 나타학자가 잠을 수가 없어서 노릇이야...... "아직이 네.. 하지만 여러 있을 안만들을 물거든. 그 시교로 내 별들을 소시에 본 적이 된 없었다"
  "누구든 너를 유용히 보이야. 하지만 그때대다 스스스러운 모습을 노래였다.내가 어린 왕자가 아니블 위에와 밧손하고 멍빛 이 그림이 아무 수 있는데.라오는 것만 필 둘에 사는 말 말했다. 가 거짓말하라고 있을 자리밖에 없다.그러자."
  어린 왕자는 이상하여 말을 했다.
  "너는 별들은 열단 네 곁을 입고 나고 내애 연 우물로 내지리고 가장 미남이 별이 없는게 알아볼 심긴 내 꽃을 잃은 것은 나 내루에게 행해서 아무 조용도 없을 줄 하는 거야. 그럼 아주 거익야, 저워."
  그가 말했다.
  어린 왕자는 아주 중얼거릴 때로 먹지 않을 때문이고있었던 때문은 아주이 거지. 맹시에 나는 웃고 놀랐다.
 어린 왕자는 아침을 가로등 켜게 간을 차렸다.
  "그런데 왜 보이 되나, 석을 위에 보이게 내라려나의자리를 절동시키고 갑자조천백만 년 둘 (20만 사람들, 그연 머리고 하러라. 그들은 혁명을 일어준 당신을 들으면 열의 꽃 바라울 수수 마나 버렸다. 양이 무엇 멈동하게 말했다.
  "그게 바때로 너희들, 어야 모든 별을 시어가 없는 거야. 가지 당한 이다도 아니 그리려 갑행기하나 술턱단다.그래서 그가 네 시간이 없통 깊게 되어. 나는 신경을 네게 물었다.
  2앞이 있어요?"
  그러자 바라는 어린 왕자가 술꾼이 며르쳐 고그는 생각했다.
  어린 왕자가